# ?? Priority #1: Measure Your Real Retrieval Ceiling
This notebook computes the **exact empirical retrieval ceiling** across all 6,686 validation questions.
It measures:
1. What fraction of rows have similarity >= 0.85, 0.70-0.85, 0.50-0.70, <0.50 per subset.
2. Exact ROUGE scores on high-similarity near-duplicate rows vs novel generation rows.
3. Identifies whether leverage is in retrieval reranking or generation editing.

In [ ]:
import os, sys, re, json, subprocess
from pathlib import Path
import pandas as pd, numpy as np, torch

try:
    _cwd = Path.cwd()
except (FileNotFoundError, OSError):
    os.chdir("/home/jovyan")
    _cwd = Path.cwd()

repo_name = "Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages"
if (_cwd / "src").exists(): BASE_DIR = _cwd
elif (_cwd.parent / "src").exists(): BASE_DIR = _cwd.parent
elif (_cwd / repo_name / "src").exists(): BASE_DIR = _cwd / repo_name
elif (Path("/home/jovyan") / repo_name / "src").exists(): BASE_DIR = Path("/home/jovyan") / repo_name
else: BASE_DIR = Path("/home/jovyan")

os.chdir(BASE_DIR)
for p in [str(BASE_DIR), str(BASE_DIR / "src")]:
    if p not in sys.path: sys.path.insert(0, p)

DATA_DIR        = BASE_DIR / "data" / "raw"
SUBMISSIONS_DIR = BASE_DIR / "submissions"
CHECKPOINTS_DIR = BASE_DIR / "models" / "checkpoints"

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

from retrieval import HybridRetriever
import importlib, rouge_utils; importlib.reload(rouge_utils)
from ceiling_measurement import measure_retrieval_ceiling

print(f"BASE_DIR: {BASE_DIR.resolve()}")


In [ ]:
train_df = pd.read_csv(DATA_DIR / "Training set.csv")
val_df   = pd.read_csv(DATA_DIR / "Validation set.csv")

print("[INFO] Initializing Hybrid RAG Retriever on Training Set (29,815 rows)...")
retriever = HybridRetriever(train_df, enable_dense=True)

print("[INFO] Running Empirical Ceiling Measurement on Validation Set (6,686 rows)...")
val_analyzed_df = measure_retrieval_ceiling(val_df, retriever)
